# IV-D 残余电导实验 (LeNet-5 / MNIST)

在 **参考端读出 (comp_pre = Table I 那套)** 之上，给清空(保护)的 crossbar 位置注入 RESET 残余电导，测它对 top-1 的影响。

**你只需要上传一个文件**: `lenet_pre_package.zip`
  （里面已经包含代码 + `data/rtl_inputs_A` + `data/rtl_inputs_A_v3`，不用再单独传 data。残余脚本由第 4 格自己写到磁盘，也不用传。）

**跑之前**: Runtime → Change runtime type → **T4 GPU**。全程约 5–8 分钟。


## 1. 确认 GPU


In [1]:
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else '!! 没有 GPU: Runtime -> Change runtime type -> T4 GPU')


torch 2.11.0+cu128 | cuda: True
NVIDIA A100-SXM4-40GB


## 2. 上传 lenet_pre_package.zip
运行后在弹窗里选 `lenet_pre_package.zip` (就一个)。


In [3]:
from google.colab import files
up = files.upload()   # 选 lenet_pre_package.zip
print('uploaded:', list(up.keys()))


Saving lenet_pre_package.zip to lenet_pre_package.zip
uploaded: ['lenet_pre_package.zip']


## 3. 解压 + 定位代码和数据
自动确认 imc_torch 是带 comp_pre 的参考端版本，并找到 data 目录。


In [4]:
import os, glob
!unzip -qo lenet_pre_package.zip
have = [f for f in ['imc_vec.py','imc_torch.py','run_lenet_gpu.py','run_lenet_gpu_pre.py'] if os.path.exists(f)]
print('base code:', have)
print('imc_torch has comp_pre (reference-side):', 'comp_pre' in open('imc_torch.py').read())
SRC = glob.glob('**/rtl_inputs_A', recursive=True)[0]
V3  = glob.glob('**/rtl_inputs_A_v3', recursive=True)[0]
print('SRC =', SRC, '| has images/img_001.hex:', os.path.exists(os.path.join(SRC,'images','img_001.hex')))
print('V3  =', V3)


base code: ['imc_vec.py', 'imc_torch.py', 'run_lenet_gpu.py', 'run_lenet_gpu_pre.py']
imc_torch has comp_pre (reference-side): True
SRC = data/rtl_inputs_A | has images/img_001.hex: True
V3  = data/rtl_inputs_A_v3


## 4. 写入残余实验脚本
这一格把 `run_lenet_resid.py` 写到磁盘。改 `RESID_SIGNED=True` 可得部分抵消(更贴近实际)的版本。


In [5]:
%%writefile run_lenet_resid.py
"""
run_lenet_resid.py  --  IV-D residual-conductance experiment.

Builds directly on run_lenet_gpu_pre.py's reference-side (comp_pre) readout, i.e.
the exact configuration that produced lenet_imc_full_pre.json / Table I. It
relaxes the single idealization that Prof. Namba flagged: that a vacated
(protected) cell contributes EXACTLY ZERO to the analog column sum.

Physical picture modeled here
-----------------------------
On the fixed PCM crossbar a protected weight is served from SRAM, and its
crossbar cells are RESET to the lowest-conductance (amorphous) state. That state
is not truly zero: it retains a small off-state conductance
    eps_cell = cmax / r          (cell-value units, cmax = 3 for a 2-bit cell)
where r is the device on/off ratio. Each of the n_mag_cells bit-slice cells of a
vacated position sits at this floor, so the residual:
  * adds a small POSITIVE current to the column sum (the SRAM path already
    supplies the exact w*x, so this is a pure additive perturbation);
  * drifts with the RESET-state exponent (reused per-cell nu ~ N(0.10,0.03);
    mean 0.10 is the amorphous/RESET drift). Drift only lowers conductance;
  * passes through the SAME comp-aware 5-bit ADC and the SAME 2^(2j) shift-add
    as the real datapath.

Conservative assumptions (this run is an UPPER BOUND)
----------------------------------------------------
  RESID_SIGNED   = False -> residual currents are placed on the + polarity column
                            and are assumed NOT to cancel across sign (worst case).
                            Set True to let them follow each vacated weight's
                            original polarity (partial cancellation, smaller).
  RESID_NU_SCALE = 1.0   -> reset-state drift exponent = model nu. Raise (e.g.
                            1.5) for a worst-case faster-drifting RESET state.

Only the 'uniform' and 'layer_aware' arms are swept: 'algo_only' has no protected
cells, hence no vacated cells and identically zero residual.

The eps=0 arm is run first and checked against the embedded Table I anchors, so a
nonzero delta can only come from the residual, not from a harness mismatch.

Usage (Colab, after `unzip lenet_pre_package.zip` and the data zip):
  python3 run_lenet_resid.py --src-dir data/rtl_inputs_A --v3-dir data/rtl_inputs_A_v3 --mnist

Output: lenet_imc_resid.json  (+ resume-safe checkpoint lenet_resid_ckpt.json)
"""
import os, json, time, argparse, math
import numpy as np
import torch
import run_lenet_gpu as G      # reuse loader, masks, forward_batch_t, nu/write-noise, COMP
import imc_torch               # we monkeypatch imc_torch.imc_matmul_t below

# ----- knobs -----
RESID_SIGNED   = False
RESID_NU_SCALE = 1.0

# ----- Table I (lenet_imc_full_pre.json) anchors for the eps=0 sanity check -----
PRE_5B = {"layer_aware": {"1e+00": 0.9819, "1e+06": 0.97972},
          "uniform":     {"1e+00": 0.95794, "1e+06": 0.92074}}
CKPT = "lenet_resid_ckpt.json"


# ============================================================
#  residual-aware replacement for imc_torch.imc_matmul_t
#  (identical to the original except for the eps floor on protected cells)
# ============================================================
def imc_matmul_t_resid(tm, L, V, cfg, kind, wn_L=None):
    cells_L, W_L, mask_L, nu_L, bias_L = tm["cells"][L], tm["W"][L], tm["mask"][L], tm["nu"][L], tm["bias"][L]
    rshift = tm["rshift"][L]
    device, dtype = W_L.device, W_L.dtype
    out_dim, K = W_L.shape
    cellBit, ncell, sub, in_bits = cfg["cellBit"], cfg["n_mag_cells"], cfg["subArray"], cfg["in_bits"]
    adc_bits, t, comp = cfg["adc_bits"], cfg["drift_t"], cfg["comp"]
    cmax = (1 << cellBit) - 1
    M = V.shape[1]
    zeroW = torch.zeros_like(W_L)

    Wp = torch.where(mask_L, W_L, zeroW)
    acc = bias_L[:, None] + Wp @ V                          # exact SRAM + protected path

    eps = float(cfg.get("resid_eps", 0.0))                  # RESET off-state floor, cell units
    Wu = torch.where(mask_L, zeroW, W_L)
    has_pcm = bool((Wu != 0).any())
    has_resid = eps > 0.0 and bool(mask_L.any())
    if not has_pcm and not has_resid:                       # e.g. conv1: fully protected, eps=0
        if kind == "raw":  return acc
        if kind == "fc3":  return imc_torch.requant_fc3_t(acc, rshift)
        return imc_torch.requant_relu_t(acc, rshift)

    signpos = (Wu > 0).to(dtype)
    signneg = (Wu < 0).to(dtype)
    if t != 1.0:
        Dpcm = torch.pow(torch.as_tensor(float(t), device=device, dtype=dtype), -nu_L)
        D = torch.where(mask_L, torch.ones_like(nu_L), Dpcm)
    else:
        D = torch.ones_like(W_L)

    # ---- residual floor on protected (vacated) positions ----
    resid_pos = resid_neg = None
    if has_resid:
        maskf = mask_L.to(dtype)
        if t != 1.0:
            Dres = torch.pow(torch.as_tensor(float(t), device=device, dtype=dtype), -nu_L * RESID_NU_SCALE)
        else:
            Dres = torch.ones_like(W_L)
        resid = eps * Dres * maskf                          # (out,K), nonzero on protected only
        if RESID_SIGNED:
            ws = torch.sign(W_L).to(dtype)
            resid_pos = resid * (ws > 0).to(dtype)
            resid_neg = resid * (ws < 0).to(dtype)
        else:
            resid_pos = resid                               # all on + column: upper bound
            resid_neg = torch.zeros_like(resid)

    T = (K + sub - 1) // sub
    Kpad = T * sub
    actual_rows = torch.full((T,), float(sub), device=device, dtype=dtype)
    if Kpad != K:
        actual_rows[-1] = float(K - (T - 1) * sub)

    def padK(a):
        if Kpad == K:  return a
        o = torch.zeros(a.shape[:-1] + (Kpad,), device=device, dtype=dtype)
        o[..., :K] = a
        return o

    Vpad = torch.zeros((Kpad, M), device=device, dtype=dtype)
    Vpad[:K, :] = V
    Vint = Vpad.to(torch.int64)
    Vbits = torch.stack([((Vint >> i) & 1).to(dtype) for i in range(in_bits)], dim=0)
    Vbits_t = Vbits.reshape(in_bits, T, sub, M)

    notmask = (~mask_L).to(dtype)
    pcm = torch.zeros((out_dim, M), device=device, dtype=dtype)
    for j in range(ncell):
        Cj = cells_L[:, :, j]
        if wn_L is not None:
            Cj = Cj + wn_L[:, :, j] * notmask
        cpos = Cj * D * signpos
        cneg = Cj * D * signneg
        if resid_pos is not None:                           # every bit-slice cell has the same floor
            cpos = cpos + resid_pos
            cneg = cneg + resid_neg
        CjD_pos = padK(cpos).reshape(out_dim, T, sub)
        CjD_neg = padK(cneg).reshape(out_dim, T, sub)
        wj = float(1 << (cellBit * j))
        for i in range(in_bits):
            Bi = Vbits_t[i]
            ppos = torch.einsum('trm,otr->otm', Bi, CjD_pos)
            pneg = torch.einsum('trm,otr->otm', Bi, CjD_neg)
            if adc_bits is not None:
                fs = (actual_rows * cmax)
                if cfg.get("comp_pre", False) and comp != 0.0 and t > 0.0:
                    fs = fs * math.exp(-comp * math.log(t))
                ppos = imc_torch.adc_quant_t(ppos, fs[None, :, None], adc_bits)
                pneg = imc_torch.adc_quant_t(pneg, fs[None, :, None], adc_bits)
            contrib = (ppos - pneg).sum(dim=1)
            bw = -(1 << i) if i == in_bits - 1 else (1 << i)
            pcm = pcm + (float(bw) * wj) * contrib
    if comp != 0.0:
        pcm = pcm * (math.exp(comp * math.log(t)) if t > 0 else 1.0)
    acc = acc + pcm

    if kind == "raw":  return acc
    if kind == "fc3":  return imc_torch.requant_fc3_t(acc, rshift)
    return imc_torch.requant_relu_t(acc, rshift)


# ============================================================
#  sweep
# ============================================================
def run_point(tm, mask_t, X, y, nu, wn, t, adc_bits, chunk, resid_eps):
    for L in G.LAYERS:
        tm["mask"][L] = mask_t[L]
    tm["nu"] = nu
    cfg = dict(G.DEFAULT_CFG)
    cfg["drift_t"] = t; cfg["comp"] = G.COMP; cfg["adc_bits"] = adc_bits
    cfg["comp_pre"] = True                       # reference-side, matches Table I
    cfg["resid_eps"] = float(resid_eps)
    correct = 0
    with torch.no_grad():
        for i in range(0, len(X), chunk):
            _, preds = G.forward_batch_t(X[i:i + chunk], tm, cfg, wn=wn)
            correct += int((preds.cpu().numpy() == y[i:i + chunk]).sum())
    return correct / len(X)


def load_ckpt():
    return json.load(open(CKPT)) if os.path.exists(CKPT) else {"cells": {}}

def save_ckpt(d):
    json.dump(d, open(CKPT + ".tmp", "w")); os.replace(CKPT + ".tmp", CKPT)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src-dir", required=True)
    ap.add_argument("--v3-dir", required=True)
    ap.add_argument("--mnist", action="store_true")
    ap.add_argument("--mnist-root", default="./_mnist")
    ap.add_argument("--n-calib", type=int, default=1000)
    ap.add_argument("--n-images", type=int, default=0)      # 0 = all 10000
    ap.add_argument("--seeds", type=int, default=5)
    ap.add_argument("--drift-times", default="1,1e1,1e2,1e3,1e4,1e5,1e6")
    ap.add_argument("--adc-bits", type=int, default=5)
    ap.add_argument("--chunk", type=int, default=1000)
    ap.add_argument("--ratios", default="inf,1000,100,10")  # on/off ratios; inf = eps=0 baseline
    ap.add_argument("--out", default="lenet_imc_resid.json")
    args = ap.parse_args()

    imc_torch.imc_matmul_t = imc_matmul_t_resid             # activate residual path

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float32
    print(f"device: {device}  | RESID_SIGNED={RESID_SIGNED}  RESID_NU_SCALE={RESID_NU_SCALE}", flush=True)

    model = G.load_model(args.src_dir, args.v3_dir)
    tm = G.to_torch_model(model, device=device, dtype=dtype)
    times = [float(x) for x in args.drift_times.split(",")]
    cmax = (1 << 2) - 1

    ratios = []
    for tok in args.ratios.split(","):
        tok = tok.strip()
        ratios.append(math.inf if tok in ("inf", "baseline", "0") else float(tok))

    sc = G.calibrate_input_scale(args.mnist_root, args.n_calib)
    X, y = G.load_images_mnist(args.n_images, sc, args.mnist_root,
                               selfcheck_hex=os.path.join(args.src_dir, "images", "img_001.hex"))
    arms = {"uniform": G.make_uniform_mask(model), "layer_aware": model["mask"]}
    arms = {k: G.masks_to_torch(v, device) for k, v in arms.items()}
    nus = [G.nu_torch(model, 1000 + s, device, dtype) for s in range(args.seeds)]
    wns = [G.writenoise_torch(model, 5000 + s, device, dtype) for s in range(args.seeds)]
    print(f"images: {len(X)}, seeds: {args.seeds}, ratios: {ratios}", flush=True)

    d = load_ckpt()
    def eps_of(r):  return 0.0 if math.isinf(r) else cmax / r
    def rtag(r):    return "inf" if math.isinf(r) else f"{r:g}"

    t0 = time.time()
    for r in ratios:
        eps = eps_of(r)
        for arm, mask_t in arms.items():
            for ro, adc in [("noADC", None), (f"{args.adc_bits}b", args.adc_bits)]:
                for t in times:
                    for s in range(args.seeds):
                        key = f"{rtag(r)}|{arm}|{ro}|{t:.0e}|s{s}"
                        if key in d["cells"]:
                            continue
                        a = run_point(tm, mask_t, X, y, nus[s], wns[s], t, adc, args.chunk, eps)
                        d["cells"][key] = a
                        save_ckpt(d)
                    seedvals = [d["cells"][f"{rtag(r)}|{arm}|{ro}|{t:.0e}|s{s}"] for s in range(args.seeds)]
                    print(f"  r={rtag(r):>4}  {arm:12s} {ro:6s} t={t:.0e}: "
                          f"{100*np.mean(seedvals):6.2f}%  [{time.time()-t0:.0f}s]", flush=True)

    # ---- assemble ----
    def agg(r, arm, ro, t):
        v = np.array([d["cells"][f"{rtag(r)}|{arm}|{ro}|{t:.0e}|s{s}"] for s in range(args.seeds)])
        return float(v.mean()), float(v.std())

    out = {"meta": {"n_images": len(X), "seeds": args.seeds, "times": times,
                    "adc_bits": args.adc_bits, "ratios": [rtag(r) for r in ratios],
                    "resid_signed": RESID_SIGNED, "resid_nu_scale": RESID_NU_SCALE,
                    "readout": "comp-aware ADC reference (comp_pre); residual eps=cmax/r on vacated cells"},
           "by_ratio": {}}
    for r in ratios:
        out["by_ratio"][rtag(r)] = {}
        for arm in arms:
            for ro in ["noADC", f"{args.adc_bits}b"]:
                for t in times:
                    m, sdv = agg(r, arm, ro, t)
                    out["by_ratio"][rtag(r)][f"{arm}|{ro}|{t:.0e}"] = {"mean": m, "std": sdv}
    json.dump(out, open(args.out, "w"), indent=2)
    print(f"\nsaved -> {args.out}", flush=True)

    # ---- eps=0 baseline sanity vs Table I ----
    print("\n=== eps=0 baseline vs Table I (must PASS before trusting deltas) ===", flush=True)
    okall = True
    for arm in ["layer_aware", "uniform"]:
        for texp in ["1e+00", "1e+06"]:
            got = agg(math.inf, arm, f"{args.adc_bits}b", float(texp))[0]
            ref = PRE_5B[arm][texp]
            ok = abs(got - ref) <= 0.001
            okall &= ok
            print(f"  {arm:12s} 5b t={texp}: got {100*got:6.2f}  Table I {100*ref:6.2f}  "
                  f"{'PASS' if ok else 'CHECK'}", flush=True)
    print(f"  baseline reproduces Table I: {'YES' if okall else 'NO -- investigate before reading deltas'}",
          flush=True)

    # ---- delta table: accuracy change vs eps=0 baseline ----
    print("\n=== top-1 change vs eps=0 baseline (pp) ===", flush=True)
    for ro in ["noADC", f"{args.adc_bits}b"]:
        print(f"\n[{ro}]", flush=True)
        header = "  arm/t      " + "".join(f"{('r='+rtag(r)):>10}" for r in ratios if not math.isinf(r))
        print(header, flush=True)
        for arm in arms:
            for t in times:
                base = agg(math.inf, arm, ro, t)[0]
                row = "".join(f"{100*(agg(r, arm, ro, t)[0]-base):>10.2f}"
                              for r in ratios if not math.isinf(r))
                print(f"  {arm[:6]:6} {t:.0e} {row}", flush=True)

    worst = max(abs(100*(agg(r, arm, ro, t)[0] - agg(math.inf, arm, ro, t)[0]))
                for r in ratios if not math.isinf(r)
                for arm in arms for ro in [f"{args.adc_bits}b"] for t in times)
    print(f"\nworst |delta| over the 5-bit grid (any swept ratio): {worst:.2f} pp", flush=True)


if __name__ == "__main__":
    main()



Writing run_lenet_resid.py


## 5. 快速自检 (2000 图 / 2 seed / r∈{inf,100}) — 几十秒
确认 **eps=0 那档和 Table I 对得上** (看 `PASS`)，管线就通了。


In [5]:
!python run_lenet_resid.py --src-dir {SRC} --v3-dir {V3} --mnist \
    --n-images 2000 --seeds 2 --ratios inf,100


device: cuda  | RESID_SIGNED=False  RESID_NU_SCALE=1.0
100% 9.91M/9.91M [00:01<00:00, 5.65MB/s]
100% 28.9k/28.9k [00:00<00:00, 133kB/s]
100% 1.65M/1.65M [00:01<00:00, 1.26MB/s]
100% 4.54k/4.54k [00:00<00:00, 17.5MB/s]
  input self-check (test[0] vs img_001.hex): PASS
images: 2000, seeds: 2, ratios: [inf, 100.0]
  r= inf  uniform      noADC  t=1e+00:  98.23%  [1s]
  r= inf  uniform      noADC  t=1e+01:  98.32%  [1s]
  r= inf  uniform      noADC  t=1e+02:  98.23%  [2s]
  r= inf  uniform      noADC  t=1e+03:  98.17%  [2s]
  r= inf  uniform      noADC  t=1e+04:  98.03%  [3s]
  r= inf  uniform      noADC  t=1e+05:  97.95%  [3s]
  r= inf  uniform      noADC  t=1e+06:  97.25%  [3s]
  r= inf  uniform      5b     t=1e+00:  94.85%  [4s]
  r= inf  uniform      5b     t=1e+01:  95.12%  [4s]
  r= inf  uniform      5b     t=1e+02:  95.97%  [5s]
  r= inf  uniform      5b     t=1e+03:  95.50%  [6s]
  r= inf  uniform      5b     t=1e+04:  95.03%  [6s]
  r= inf  uniform      5b     t=1e+05:  94.17%  [7s

## 6. 完整跑 (全部 10000 图 / 5 seed / r∈{inf,1000,100,10})
T4 上约 5–8 分钟。输出末尾就是 IV-D 要的: eps=0 vs Table I 的 PASS、每档 r 相对基线的 top-1 变化 (pp)、5-bit 网格最大 |Δ|。


In [6]:
!python run_lenet_resid.py --src-dir {SRC} --v3-dir {V3} --mnist \
    --n-images 0 --seeds 5 --ratios inf,1000,100,10 --out lenet_imc_resid.json


device: cuda  | RESID_SIGNED=False  RESID_NU_SCALE=1.0
  input self-check (test[0] vs img_001.hex): PASS
images: 10000, seeds: 5, ratios: [inf, 1000.0, 100.0, 10.0]
  r= inf  uniform      noADC  t=1e+00:  98.54%  [3s]
  r= inf  uniform      noADC  t=1e+01:  98.56%  [6s]
  r= inf  uniform      noADC  t=1e+02:  98.46%  [8s]
  r= inf  uniform      noADC  t=1e+03:  98.36%  [11s]
  r= inf  uniform      noADC  t=1e+04:  98.10%  [13s]
  r= inf  uniform      noADC  t=1e+05:  97.78%  [16s]
  r= inf  uniform      noADC  t=1e+06:  96.94%  [19s]
  r= inf  uniform      5b     t=1e+00:  95.39%  [23s]
  r= inf  uniform      5b     t=1e+01:  95.77%  [28s]
  r= inf  uniform      5b     t=1e+02:  95.75%  [33s]
  r= inf  uniform      5b     t=1e+03:  95.41%  [37s]
  r= inf  uniform      5b     t=1e+04:  94.90%  [42s]
  r= inf  uniform      5b     t=1e+05:  93.99%  [47s]
  r= inf  uniform      5b     t=1e+06:  91.92%  [51s]
  r= inf  layer_aware  noADC  t=1e+00:  98.64%  [53s]
  r= inf  layer_aware  noADC

## 7. 下载结果


In [7]:
from google.colab import files
files.download('lenet_imc_resid.json')
# 断线重连也不怕: 进度存在 lenet_resid_ckpt.json, 重跑第 6 格会接着来


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
!ls

sample_data


In [6]:
!sed -i 's/RESID_SIGNED *= *False/RESID_SIGNED   = True/' run_lenet_resid.py
!rm -f lenet_resid_ckpt.json
!python run_lenet_resid.py --src-dir {SRC} --v3-dir {V3} --mnist \
    --n-images 0 --seeds 5 --ratios inf,10,100 --out lenet_imc_resid_signed.json

device: cuda  | RESID_SIGNED=True  RESID_NU_SCALE=1.0
100% 9.91M/9.91M [00:01<00:00, 5.68MB/s]
100% 28.9k/28.9k [00:00<00:00, 131kB/s]
100% 1.65M/1.65M [00:01<00:00, 1.26MB/s]
100% 4.54k/4.54k [00:00<00:00, 16.9MB/s]
  input self-check (test[0] vs img_001.hex): PASS
images: 10000, seeds: 5, ratios: [inf, 10.0, 100.0]
  r= inf  uniform      noADC  t=1e+00:  98.75%  [5s]
  r= inf  uniform      noADC  t=1e+01:  98.76%  [9s]
  r= inf  uniform      noADC  t=1e+02:  98.68%  [14s]
  r= inf  uniform      noADC  t=1e+03:  98.57%  [18s]
  r= inf  uniform      noADC  t=1e+04:  98.30%  [23s]
  r= inf  uniform      noADC  t=1e+05:  97.92%  [27s]
  r= inf  uniform      noADC  t=1e+06:  97.11%  [31s]
  r= inf  uniform      5b     t=1e+00:  95.79%  [39s]
  r= inf  uniform      5b     t=1e+01:  96.10%  [47s]
  r= inf  uniform      5b     t=1e+02:  96.02%  [54s]
  r= inf  uniform      5b     t=1e+03:  95.77%  [62s]
  r= inf  uniform      5b     t=1e+04:  95.19%  [70s]
  r= inf  uniform      5b     t=1e+

In [7]:
from google.colab import files
files.download('lenet_imc_resid_signed.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>